# 06 - Démonstration sur des images inconnues

Le système complet tel qu'il tournerait à bord : **deux fichiers ONNX en 8 bits et leurs seuils**, rien
d'autre. Pas de PyTorch, pas de GPU, pas d'annotation.

- modèle principal, **optique + radar** : `modeles/fusion/one_look_fusion_int8.onnx` (notebook 08) ;
- modèle de **secours, optique seul** : `modeles/one_look_int8.onnx` (notebook 05), utilisé dès que
  l'image radar manque ou est inexploitable.

Images utilisées : pays **jamais vus à l'entraînement ni pour régler les seuils** (Bolivie, Espagne,
Ghana), puis des anomalies simulées. À la fin, une petite application permet de déposer n'importe
quelle image Sentinel-2 (GeoTIFF à 13 bandes), avec ou sans l'image radar Sentinel-1.

Exécution : Colab, *Exécution > Tout exécuter* (environ 2 minutes, pas besoin de GPU).

In [ ]:
import os, subprocess, sys

if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
os.chdir("one-look")
sys.path.insert(0, ".")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "onnxruntime", "gradio"], check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import demo, donnees

assert os.path.exists("modeles/config.json"), "modèle absent : lancer d'abord le notebook 05"
systeme = demo.OneLook("modeles")
for nom, m in [("principal (fusion)", systeme.fusion), ("secours (optique)", systeme.optique)]:
    if m is None:
        print(nom, ": absent (lancer le notebook 08) ; seul le modèle optique sera utilisé")
    else:
        print(f"Modèle {nom} : {m.chemin_modele} ({os.path.getsize(m.chemin_modele) / 1e6:.2f} Mo), seuils {m.seuils}")

## 1. Récupérer quelques images de test

Images Sentinel-2 brutes, images radar Sentinel-1 du même jour et carte d'eau permanente (connue à l'avance, c'est une carte fixe).
Les annotations ne servent **qu'à vérifier** après coup : le système ne les voit jamais.

In [ ]:
EXEMPLES = ["Bolivia_103757", "Bolivia_360519", "Spain_7370579", "Ghana_313799", "Ghana_866994", "Ghana_26376"]
BUCKET = donnees.BUCKET
os.makedirs("exemples", exist_ok=True)
for nom in EXEMPLES:
    for dossier in ["S2Hand", "S1Hand", "JRCWaterHand", "LabelHand"]:
        cible = f"exemples/{nom}_{dossier}.tif"
        if not os.path.exists(cible):
            subprocess.run(["gsutil", "-q", "cp", f"{BUCKET}/{dossier}/{nom}_{dossier}.tif", cible])
EXEMPLES = [n for n in EXEMPLES if os.path.exists(f"exemples/{n}_S2Hand.tif")]
print(len(EXEMPLES), "images :", EXEMPLES)

## 2. Le système face à des images jamais vues

Pour chaque image : la décision du système complet (fusion), puis, pour comparer, ce qu'aurait dit le
modèle optique seul.

In [ ]:
lignes = []
for nom in EXEMPLES:
    s2 = demo.lire_tif(f"exemples/{nom}_S2Hand.tif")
    perm = demo.lire_tif(f"exemples/{nom}_JRCWaterHand.tif")[0]
    label = demo.lire_tif(f"exemples/{nom}_LabelHand.tif")[0]
    chemin_s1 = f"exemples/{nom}_S1Hand.tif"
    s1 = demo.lire_tif(chemin_s1) if os.path.exists(chemin_s1) else None
    res = systeme.analyser(s2, perm, s1)
    seul = systeme.analyser(s2, perm)                      # sans radar : modèle de secours
    vrai = donnees.part_inondee_vraie(label, perm == 1)
    demo.figure(s2, res, nom); plt.show()
    print(demo.resume(res))
    print(f"Optique seul : {seul['decision']} ({seul['part_eau']:.1%} d'eau anormale)")
    print(f"Vérité (annotation) : {vrai:.1%} inondé → {'alerte attendue' if vrai >= 0.05 else 'pas d’alerte attendue'}\n")
    lignes.append({"image": nom, "modele": res["modele"], "decision": res["decision"],
                   "eau_detectee": round(res["part_eau"], 3), "optique_seul": seul["decision"],
                   "eau_optique_seul": round(seul["part_eau"], 3),
                   "inonde_vrai": round(float(vrai), 3), "nuages": round(res["nuages"], 2),
                   "temps_ms": round(res["temps_ms"])})
pd.DataFrame(lignes)

## 3. Anomalies simulées

Un capteur qui sature, des bandes infrarouges perdues, du bruit : le système doit refuser de
conclure (**INCONNU**) au lieu de lancer une fausse alerte. Si c'est le **radar** qui est perdu, le
système bascule sur le modèle optique de secours et continue de répondre.

In [ ]:
nom = EXEMPLES[0]
s2 = demo.lire_tif(f"exemples/{nom}_S2Hand.tif")
perm = demo.lire_tif(f"exemples/{nom}_JRCWaterHand.tif")[0]
chemin_s1 = f"exemples/{nom}_S1Hand.tif"
s1 = demo.lire_tif(chemin_s1) if os.path.exists(chemin_s1) else None
anomalies = {
    "originale": s2,
    "capteur saturé (× 3)": s2 * 3,
    "bandes infrarouges perdues": np.concatenate([s2[:7], np.zeros_like(s2[7:])]),
    "bruit": np.random.default_rng(0).uniform(0, 10000, s2.shape).astype(np.float32),
}
lignes = []
for t, x in anomalies.items():
    r = systeme.analyser(x, perm, s1)
    lignes.append({"transformation": t, "modele": r["modele"], "decision": r["decision"],
                   "score_inconnu": round(r["score_inconnu"], 1), "seuil_inconnu": r["seuils"]["inconnu"]})
if s1 is not None:
    r = systeme.analyser(s2, perm, np.full_like(s1, np.nan))
    lignes.append({"transformation": "radar perdu", "modele": r["modele"], "decision": r["decision"],
                   "score_inconnu": round(r["score_inconnu"], 1), "seuil_inconnu": r["seuils"]["inconnu"]})
print(f"Image de départ : {nom}")
pd.DataFrame(lignes)

## 4. Temps de calcul sur un seul cœur de processeur

Image complète 512 × 512 pixels, réseau + décision.

In [ ]:
import time
x = demo.lire_tif(f"exemples/{EXEMPLES[0]}_S2Hand.tif")
chemin_s1 = f"exemples/{EXEMPLES[0]}_S1Hand.tif"
r1 = demo.lire_tif(chemin_s1) if os.path.exists(chemin_s1) else None
for nom, radar in [("optique + radar", r1), ("optique seul", None)]:
    r = systeme.analyser(x, None, radar)  # échauffement
    t = [systeme.analyser(x, None, radar)["temps_ms"] for _ in range(5)]
    print(f"{r['modele']:24s} {np.mean(t):.0f} ms par image (min {np.min(t):.0f}, max {np.max(t):.0f})")

## 5. Application de démonstration

Déposer une image Sentinel-2 (GeoTIFF à 13 bandes, niveau L1C, comme Sen1Floods11), si possible
l'image radar Sentinel-1 (VV et VH en dB) et la carte d'eau permanente, puis simuler une anomalie ou
la perte du radar. Les images de test ci-dessus sont proposées en
exemples.

In [ ]:
import runpy
app = runpy.run_path("app/app.py", run_name="demo")["app"]
app.launch(share=False, inline=True, height=850)